# Cross-Tile Spatial Relation Reasoning for Ultra-High-Resolution Remote SensingEnd-to-end driver for the study:> *Does explicit cross-tile spatial relation representation improve vision-language> model performance on relational reasoning tasks in ultra-high-resolution remote> sensing imagery?***Run this on the GPU machine.** Each section maps to a section of the research plan.| Notebook section | Plan section | What it produces ||---|---|---|| 1. Environment | 14.1 | preflight report || 2. Data | 1 | normalised samples, task counts || 3. Inspect one scene | 5.1-5.4 | tiling, selection, objects, relations for one image || 4. Run the arms | 6 | `predictions.jsonl` per arm || 5. Aggregate | 7, 9 | accuracy, compute and significance tables || 6. Figures | 15 | the paper's figures || 7. Error analysis | 10 | error taxonomy, detection-quality association || 8. Qualitative | 10 | scene-graph overlays, case cards || 9. Ablations | 8 | component ablation table || 10. Flood case | 11 | qualitative application demonstration |**The experimental design in one line:** Baseline 3 and the Proposed method are thesame class (`HierarchicalPipeline`) differing by one boolean, `use_relations`. Sametiles, same selection, same VLM calls, same pixels. Only the relation block differs.**Before you trust any number**, check that `backend_is_stub` is `False` in the runmanifests. The stub backend produces placeholder text so the pipeline can be validatedwithout a GPU; the aggregator refuses stub runs unless `--allow-stub` is passed.

## 0. SetupRun from the repository root, or adjust `PROJECT_ROOT` below.

In [ ]:
import json, os, sys, timefrom pathlib import Path# Point this at the Codebase directory if the notebook is run from elsewhere.PROJECT_ROOT = Path.cwd()if not (PROJECT_ROOT / "ctsr").exists():    PROJECT_ROOT = PROJECT_ROOT.parentassert (PROJECT_ROOT / "ctsr").exists(), (    "Could not locate the ctsr package. Set PROJECT_ROOT manually.")sys.path.insert(0, str(PROJECT_ROOT))os.chdir(PROJECT_ROOT)print("Project root:", PROJECT_ROOT)from ctsr.config import build_backend, load_config, load_samples, relational_tasksfrom ctsr.utils.logging_utils import get_logger, set_global_levelfrom ctsr.utils.seed import set_seedlog = get_logger("notebook")set_seed(1234)# --- choose your configuration -------------------------------------------# configs/dev.yaml   : 8 samples per sub-task, fast GPU iteration# configs/main.yaml  : the full experiment# configs/low_vram.yaml : reduced pixel budget for a ~24 GB GPU# configs/smoke.yaml : NO GPU, placeholder answers, plumbing check onlyCONFIG = "configs/dev.yaml"cfg = load_config(CONFIG)RESULTS_DIR = Path(cfg["run"]["out_dir"])REPORT_DIR = RESULTS_DIR / "report"print("Config :", CONFIG)print("Backend:", cfg["model"]["backend"], "|", cfg["model"]["model_id"])print("Results:", RESULTS_DIR)

## 1. Environment preflight (plan 14.1)Catches the problems that would otherwise surface eleven hours into a run.

In [ ]:
import subprocessres = subprocess.run([sys.executable, "scripts/00_check_env.py", "--config", CONFIG],                     capture_output=True, text=True)print(res.stdout[-6000:])if res.returncode != 0:    print("\n!! Required checks FAILED - fix these before running the experiments.")    print(res.stderr[-3000:])

## 2. Data (plan 1, 7)Loads XLRS-Bench through the adapter. The loader targets the verified lite-splitschema (`path`, `index`, `question`, `multi-choice options`, `answer`, `category`,`l2-category`, `image`) and falls back to alias matching; if a column still cannot befound it raises with the actual column list rather than guessing.Check the printed partition before going further: the relational / non-relationalsplit is **our** pre-registered grouping, not part of the benchmark, and the twogroup-level tests are the confirmatory analysis.

In [ ]:
samples, dataset_info = load_samples(cfg)rel_tasks = relational_tasks(cfg)print("Loaded {} samples".format(len(samples)))print("Multi-image (bi-temporal):", dataset_info["n_multi_image"])print("Unscorable (no answer):   ", dataset_info["n_unscorable"])print("\nBy task family:", dataset_info["by_family"])print("\nBy relational group:", dataset_info["by_relational_group"])if dataset_info.get("unrecognised_categories"):    print("\n!! Unrecognised category labels (grouped as 'unknown', excluded from")    print("   BOTH test arms):", dataset_info["unrecognised_categories"])    print("   Add aliases in ctsr/data/taxonomy.py if these are known sub-tasks.")print("\nPer sub-task:")for task, n in sorted(dataset_info["by_task"].items(), key=lambda kv: -kv[1]):    mark = "R" if task in rel_tasks else "N"    print("  [{}] {:<44} {}".format(mark, task, n))

### A sample, as the pipeline sees it

In [ ]:
s = samples[0]print("uid      :", s.uid)print("task     :", s.task, "|", s.l2, "|", s.family)print("relational:", s.rel_group())print("images   :", s.image_paths)print("question :", s.question[:300])if s.options:    print("options  :")    print(s.options_block())print("answer   :", repr(s.answer_raw)[:200])

## 3. Inspect one scene (plan 5.1-5.4)Runs the four stages on a single image so you can see what the model is actuallygiven, before committing GPU-hours. This is also the fastest way to sanity-check anew tile size or selection setting.

In [ ]:
from ctsr.tiling.tiler import TiledImagewith TiledImage(s.image_path,                tile_size=cfg["tiling"]["tile_size"],                overlap=cfg["tiling"]["overlap"],                max_tiles=cfg["tiling"]["max_tiles"]) as ti:    print(json.dumps(ti.describe(), indent=2))    overview = ti.overview(cfg["tiling"]["overview_side"])    print("\nOverview size:", overview.size,          "| downscale factor: {:.1f}x".format(ti.width / overview.size[0]))try:    import matplotlib.pyplot as plt    fig, ax = plt.subplots(figsize=(6, 6))    ax.imshow(overview); ax.axis("off")    ax.set_title("Global overview - what a direct call effectively sees")    plt.show()except ImportError:    print("matplotlib not installed; skipping the preview")

### 3.1 Load the modelLoaded once and reused by every stage and every arm, so no arm sees different weights.

In [ ]:
t0 = time.time()backend = build_backend(cfg)backend.load()print("Loaded in {:.1f}s".format(time.time() - t0))print(json.dumps(backend.describe(), indent=2, default=str))if getattr(backend, "is_stub", False):    print("\n!! STUB BACKEND - output is placeholder text, not results.")

### 3.2 Question-conditioned tile selection (plan 5.2)

In [ ]:
from ctsr.tiling.selection import build_scorer, question_keywords, select_tilesscorer = build_scorer(cfg["selection"], backend)keywords = question_keywords(s.question, s.options)print("Question keywords:", keywords)with TiledImage(s.image_path, tile_size=cfg["tiling"]["tile_size"],                overlap=cfg["tiling"]["overlap"],                max_tiles=cfg["tiling"]["max_tiles"]) as ti:    scores = scorer.score(ti, ti.tiles, s.question, s.options)    chosen = select_tiles(scores, ti.tiles, top_k=cfg["selection"]["top_k"],                          spatial_diversity=cfg["selection"]["spatial_diversity"],                          diversity_radius_frac=cfg["selection"]["diversity_radius_frac"],                          image_size=(ti.width, ti.height))    print("\n{} tiles scored, top {} selected: {}".format(        len(scores), len(chosen), [t.tile_id for t in chosen]))    print("\nTop 8 by score:")    for sc in sorted(scores, key=lambda x: -x.score)[:8]:        print("  tile {:>3}  score {:.4f}  {}".format(            sc.tile_id, sc.score,            {k: round(v, 3) for k, v in sc.components.items()}))    selected_tiles = list(chosen)    img_size = (ti.width, ti.height)

### 3.3 Local extraction and the cross-tile scene graph (plan 5.3, 5.4)Boxes are requested in **crop-relative** 0-1000 coordinates and lifted to globalpixels in code. Asking the model to do the coordinate arithmetic itself is a knownfailure mode; doing it in `ctsr.spatial.coords` is exact.

In [ ]:
from ctsr.models.detector import build_proposerfrom ctsr.spatial.dedup import cap_objects, merge_detectionsfrom ctsr.spatial.relations import RelationConfig, build_relationsfrom ctsr.spatial.scene_graph import SceneGraphfrom ctsr.utils.compute import ComputeMeterfrom ctsr.utils.errors import ErrorCollectorproposer = build_proposer(cfg["detection"], backend)meter, collector = ComputeMeter(), ErrorCollector()with TiledImage(s.image_path, tile_size=cfg["tiling"]["tile_size"],                overlap=cfg["tiling"]["overlap"],                max_tiles=cfg["tiling"]["max_tiles"]) as ti:    extraction = proposer.extract(ti, selected_tiles, s.question,                                  keywords=keywords, meter=meter, collector=collector)    print("Extraction:", json.dumps(extraction.to_dict(), indent=2))    objects, merge_stats = merge_detections(        extraction.objects, image_size=img_size,        iou_thresh=cfg["dedup"]["iou_thresh"],        cross_tile_iou_thresh=cfg["dedup"]["cross_tile_iou_thresh"],        containment_thresh=cfg["dedup"]["containment_thresh"],        seam_gap_px=cfg["dedup"]["seam_gap_px"])    print("\nDeduplication:", json.dumps(merge_stats.to_dict(), indent=2))    objects = cap_objects(objects, cfg["detection"]["max_objects_total"], keywords)    relations = build_relations(objects, img_size, RelationConfig(        max_neighbours=cfg["relations"]["max_neighbours"],        adjacency_gap_frac=cfg["relations"]["adjacency_gap_frac"],        near_dist_frac=cfg["relations"]["near_dist_frac"],        use_compass=cfg["relations"]["use_compass"],        gsd_m_per_px=cfg["relations"]["gsd_m_per_px"],        max_relations=cfg["relations"]["max_relations"]))graph = SceneGraph(objects=objects, relations=relations, image_size=img_size)print("\nGraph statistics:", json.dumps(graph.stats(), indent=2))print("\nCompute so far:", json.dumps(meter.to_dict(), indent=2))

**The two prompt blocks.** The object block goes to both arms; the relation blockgoes only to the proposed arm. This is the entire experimental manipulation.

In [ ]:
print("=" * 78)print("OBJECT BLOCK  (given to BOTH Baseline 3 and the Proposed method)")print("=" * 78)for line in graph.object_lines()[:20]:    print(" ", line)print()print("=" * 78)print("RELATION BLOCK  (given ONLY to the Proposed method)")print("=" * 78)for line in graph.relation_lines()[:25]:    print(" ", line)n_cross = sum(1 for r in graph.relations if r.cross_tile)print("\n{} of {} relations span tiles ({:.0%}).".format(    n_cross, graph.n_relations, n_cross / max(1, graph.n_relations)))print("Those are exactly the relations independent tiling cannot represent.")

### 3.4 Visualise the scene graph

In [ ]:
from ctsr.report.qualitative import overlay_scene_graphpseudo_record = {    "uid": s.uid, "task": s.task, "n_tiles_total": None,    "artifacts": {        "scene_graph": graph.to_dict(),        "selection": [{"chosen": [t.tile_id for t in selected_tiles]}],        "tile_size": cfg["tiling"]["tile_size"],        "overlap": cfg["tiling"]["overlap"],    },}paths = overlay_scene_graph(pseudo_record, s.image_path, REPORT_DIR / "explore")print("Wrote:", paths)try:    from IPython.display import Image as IPImage, display    if paths:        display(IPImage(filename=[p for p in paths if p.endswith(".png")][0]))except Exception as exc:    print("(preview unavailable:", exc, ")")

## 4. Run the experimental arms (plan 6)Four arms, in order. Each writes `<out_dir>/<arm>/predictions.jsonl` and **resumes**from whatever is already there, so re-running this cell after an interruption picks upwhere it stopped rather than starting over.Expect this to be the long part. Rough cost per sample:| Arm | VLM calls ||---|---|| `b1_direct` | 1 || `b2_tiling` | `max_answer_tiles` + 1 || `b3_selected` | `top_k` + 1 || `proposed_ctsr` | `top_k` + 1 (identical to B3) |If `b3_selected` and `proposed_ctsr` do not end up with the *same* mean call count,the comparison is not compute-matched and something in the config differs betweenthem - section 5 checks this explicitly.

In [ ]:
from ctsr.pipelines.runner import build_pipeline, run_experimentARMS = cfg["run"]["arms"]        # or e.g. ["b3_selected", "proposed_ctsr"]LIMIT = None                     # set to a small int for a quick passsummaries = {}for arm in ARMS:    print("\n" + "=" * 78)    print("ARM:", arm)    print("=" * 78)    pipeline = build_pipeline(arm, cfg, backend, scorer=scorer, proposer=proposer)    summaries[arm] = run_experiment(        pipeline=pipeline, samples=samples, out_dir=RESULTS_DIR, run_name=arm,        resume=cfg["run"]["resume"], limit=LIMIT, cfg=cfg,        dataset_info=dataset_info, progress_every=cfg["run"]["progress_every"])    print(json.dumps(summaries[arm], indent=2, default=str))

## 5. Aggregate: accuracy, compute and significance (plan 7, 9)

In [ ]:
from ctsr.eval.aggregate import ARM_LABELS, aggregate_allagg = aggregate_all(    str(RESULTS_DIR),    baseline_arm="b3_selected", proposed_arm="proposed_ctsr",    relational=rel_tasks,    allow_stub=bool(getattr(backend, "is_stub", False)),    n_boot=cfg["eval"]["n_bootstrap"])cov = agg["coverage"]print("Samples per arm:", cov["n_per_arm"])print("Paired across all arms:", cov["n_common"])for w in cov.get("warnings", []):    print("  !", w)

In [ ]:
print("ACCURACY (%)  [95% bootstrap CI]")print("-" * 92)print("{:<46} {:>14} {:>14} {:>14}".format("Method", "Overall", "Relational", "Non-relational"))for r in agg["main"]:    def cell(k):        v, lo, hi = r.get(k + "_value"), r.get(k + "_ci_lo"), r.get(k + "_ci_hi")        if v is None:            return "--"        return "{:.1f} [{:.0f},{:.0f}]".format(100*v, 100*lo, 100*hi) if lo is not None \               else "{:.1f}".format(100*v)    print("{:<46} {:>14} {:>14} {:>14}".format(        ARM_LABELS.get(r["arm"], r["arm"])[:46],        cell("overall"), cell("relational"), cell("non_relational")))

### 5.1 The compute-match checkThis must pass for the central claim to hold. Baseline 3 and the proposed methodshould issue the **same** number of VLM calls and encode the **same** pixels,differing only in prompt tokens - which is the relation block's true cost.

In [ ]:
comp = {r["arm"]: r for r in agg["compute"]}b, p = comp.get("b3_selected"), comp.get("proposed_ctsr")if b and p:    rows = [("VLM calls / sample", "mean_vlm_calls", "{:.2f}"),            ("Megapixels encoded", "mean_megapixels", "{:.2f}"),            ("Prompt tokens",      "mean_prompt_tokens", "{:.0f}"),            ("Generated tokens",   "mean_completion_tokens", "{:.0f}"),            ("Seconds / sample",   "mean_seconds", "{:.1f}")]    print("{:<22} {:>14} {:>14}".format("", "Baseline 3", "Proposed"))    print("-" * 52)    for label, key, fmt in rows:        bv, pv = b.get(key), p.get(key)        print("{:<22} {:>14} {:>14}".format(            label,            fmt.format(bv) if bv is not None else "--",            fmt.format(pv) if pv is not None else "--"))    if b.get("mean_vlm_calls") and p.get("mean_vlm_calls"):        gap = abs(b["mean_vlm_calls"] - p["mean_vlm_calls"])        if gap <= 0.05 * max(b["mean_vlm_calls"], 1):            print("\nCOMPUTE-MATCHED: call counts agree to within 5%.")        else:            print("\n!! NOT compute-matched: call counts differ by {:.2f}.".format(gap))            print("   Check that both arms used the same selection and detection settings.")else:    print("Both arms are needed for the compute-match check.")

### 5.2 Confirmatory testsTwo pre-registered, paired tests (McNemar exact), reported uncorrected. The 16per-sub-task tests are exploratory and Holm-corrected.The hypothesis predicts a **gain on relational tasks and little change elsewhere** -so a uniform gain across both groups would *not* support the specific claim.

In [ ]:
sig = agg["significance"]if sig.get("note"):    print("!", sig["note"])for group in ("relational", "non_relational"):    r = sig.get("primary", {}).get(group, {})    print("\n" + "=" * 62)    print(group.upper().replace("_", "-"))    print("=" * 62)    if not r or r.get("n_pairs") in (None, 0):        print("  no paired samples")        continue    print("  paired samples : {}".format(r["n_pairs"]))    print("  Baseline 3     : {:.1f}%".format(100 * r["mean_a"]))    print("  Proposed       : {:.1f}%".format(100 * r["mean_b"]))    ci = " [{:+.1f}, {:+.1f}]".format(100*r["ci_lo"], 100*r["ci_hi"]) \         if r.get("ci_lo") is not None else ""    print("  difference     : {:+.1f} pp{}".format(100 * r["delta"], ci))    print("  test           : {}".format(r.get("test")))    print("  p-value        : {:.4g}   {}".format(        r["p_value"], "SIGNIFICANT at 0.05" if r["p_value"] < 0.05 else "not significant"))    print("  discordant     : {} wins / {} losses".format(        r.get("b01_only_b_correct"), r.get("b10_only_a_correct")))

### 5.3 Per-sub-task detail (exploratory, Holm-corrected)

In [ ]:
print("{:<42} {:>6} {:>9} {:>9} {:>10}".format("Sub-task", "grp", "delta", "p", "p(Holm)"))print("-" * 82)for task, r in sorted(sig.get("per_task", {}).items()):    if not isinstance(r, dict) or r.get("delta") is None:        continue    grp = "R" if task in rel_tasks else "N"    padj = r.get("p_adjusted")    print("{:<42} {:>6} {:>9} {:>9} {:>10} {}".format(        task[:42], grp,        "{:+.1f}".format(100 * r["delta"]),        "{:.3g}".format(r["p_value"]) if r.get("p_value") is not None else "--",        "{:.3g}".format(padj) if padj is not None else "--",        "*" if (padj is not None and padj < 0.05) else ""))

### 5.4 Descriptive result: how much structure spans tilesThis stands on its own regardless of the accuracy outcome: it quantifies the relational structure that independent tiling structurally cannot see.

In [ ]:
diag = {r["arm"]: r for r in agg["diagnostics"]}d = diag.get("proposed_ctsr", {})for label, key in [("objects per scene", "mean_objects"),                   ("relations per scene", "mean_relations_built"),                   ("cross-tile relations", "mean_cross_tile_relations"),                   ("cross-tile fraction", "mean_cross_tile_fraction"),                   ("objects merged across tiles", "mean_cross_tile_merges"),                   ("duplicate detection rate", "mean_duplicate_rate"),                   ("tiles selected", "mean_tiles_selected"),                   ("tiles available", "mean_tiles_total")]:    v = d.get(key)    print("  {:<32} {}".format(label, "{:.3f}".format(v) if v is not None else "--"))

## 6. Tables and figures (plan 15)

In [ ]:
from ctsr.report import figures as figmodfrom ctsr.report import tables as tabmodtable_paths = tabmod.write_all(agg, REPORT_DIR, pct=True)print("Tables:")for name, paths in table_paths.items():    print("  {:<14} {}".format(name, paths))

In [ ]:
from ctsr.report.error_taxonomy import build_taxonomytry:    taxonomy = build_taxonomy(str(RESULTS_DIR), arm="proposed_ctsr")except Exception as exc:    print("Error taxonomy unavailable:", exc)    taxonomy = Nonefig_paths = figmod.write_all(agg, REPORT_DIR, results_dir=str(RESULTS_DIR),                             taxonomy=taxonomy, pct=True)print("Figures:")for name, paths in fig_paths.items():    print("  {:<26} {}".format(name, paths))

In [ ]:
from IPython.display import Image as IPImage, displayfor stem in ["fig1_main_comparison", "fig2_per_task_delta", "fig4_progression",             "fig3_accuracy_vs_compute", "fig5_cross_tile_structure"]:    p = REPORT_DIR / (stem + ".png")    if p.exists():        print("\n" + stem)        display(IPImage(filename=str(p)))

## 7. Error analysis (plan 10)Failures are classified into the plan's categories, assigned in priority order so eachfailure counts exactly once. The association between detection quality and finalaccuracy is reported as an association, not an effect: detection quality is notmanipulated here.

In [ ]:
if taxonomy:    print("Scored failures:", taxonomy["n_failed"],          "| excluded as unscorable:", taxonomy["n_unscorable_excluded"])    print("\nCategories:")    for cat, n in taxonomy["counts"].items():        frac = taxonomy["fractions"].get(cat, 0)        print("  {:<28} {:>4}  ({:.0%})".format(cat.replace("_", " "), n, frac))    print("\nBy task group:")    for grp, counts in taxonomy["by_group"].items():        print("  {}: {}".format(grp, counts))    print("\nExample failures:")    for cat, examples in list(taxonomy["examples"].items())[:3]:        print("\n  [{}]".format(cat))        for ex in examples[:2]:            print("    {} | {} | pred={} gt={}".format(                ex["uid"], ex["task"], ex["prediction"], ex["gt"]))

In [ ]:
from ctsr.report.error_taxonomy import detection_quality_correlationtry:    corr = detection_quality_correlation(str(RESULTS_DIR), arm="proposed_ctsr")    print("Accuracy by number of objects recovered:")    for band, v in corr["accuracy_by_n_objects"].items():        print("  {:>7} objects: n={:<5} accuracy={:.3f}".format(band, v["n"], v["accuracy"]))    print("\ncorr(n_objects, correct)              = {}".format(        corr["corr_n_objects_vs_correct"]))    print("corr(extraction parse failure, correct) = {}".format(        corr["corr_extraction_parse_failure_vs_correct"]))    print("\n" + corr["note"])except Exception as exc:    print("Unavailable:", exc)

## 8. Qualitative analysis (plan 10)Wins, losses **and** shared failures. A qualitative section showing only successes isnot evidence, so losses are generated deliberately.

In [ ]:
from ctsr.report.qualitative import pick_interesting_cases, write_qualitative_setimage_lookup = {s.uid: s.image_paths[0] for s in samples if s.image_paths}picks = pick_interesting_cases(str(RESULTS_DIR), n_each=3)for kind, uids in picks.items():    print("{:<12} {}".format(kind, uids))made = write_qualitative_set(str(RESULTS_DIR), REPORT_DIR / "qualitative",                             image_lookup=image_lookup, n_each=3)print("\nCase cards:", sum(len(v) for v in made["cases"].values()))print("Figures    :", len(made["figures"]))

In [ ]:
# Inspect one win in detail: what did the relation block supply that Baseline 3 lacked?from ctsr.utils.io_utils import load_jsonlwins = picks.get("wins") or []if wins:    uid = wins[0]    b3 = {r["uid"]: r for r in load_jsonl(RESULTS_DIR / "b3_selected" / "predictions.jsonl")}    pr = {r["uid"]: r for r in load_jsonl(RESULTS_DIR / "proposed_ctsr" / "predictions.jsonl")}    a, b = b3.get(uid, {}), pr.get(uid, {})    print("uid      :", uid)    print("task     :", b.get("task"), "|", b.get("rel_group"))    print("ground truth:", b.get("gt"))    print("\nBaseline 3 ->", a.get("parsed"), "(correct:", a.get("correct"), ")")    print("Proposed   ->", b.get("parsed"), "(correct:", b.get("correct"), ")")    g = (b.get("artifacts") or {}).get("scene_graph", {})    cross = [r for r in g.get("relations", []) if r.get("cross_tile")]    print("\n{} cross-tile relations were available to the proposed arm:".format(len(cross)))    objs = {o["obj_id"]: o["label"] for o in g.get("objects", [])}    for r in cross[:12]:        print("  #{} {} --{}--> #{} {}".format(            r["subject"], objs.get(r["subject"], "?"), r["predicate"],            r["object"], objs.get(r["object"], "?")))else:    print("No wins found in this run.")

## 9. Ablations (plan 8)Each variant changes one component and holds tiles, selection, model and call budgetfixed. Run them from the command line - each is a full pass over the data:```bashfor c in configs/experiments/a*.yaml; do    python scripts/02_run_experiment.py --config "$c"done```Sweeps over tile size, overlap, selected-region count and pixel budget live in`configs/experiments/s_*.yaml`. Then roll everything up:

In [ ]:
import subprocessABLATION_ROOT = "results/ablation"if Path(ABLATION_ROOT).exists():    res = subprocess.run(        [sys.executable, "scripts/03_aggregate_results.py",         "--results", str(RESULTS_DIR), "--out", str(REPORT_DIR),         "--ablation-root", ABLATION_ROOT, "--config", CONFIG],        capture_output=True, text=True)    print(res.stdout[-4000:])else:    print("No ablation results yet at {}. Run the configs above first.".format(ABLATION_ROOT))    print("\nAvailable ablation configs:")    for p in sorted(Path("configs/experiments").glob("a*.yaml")):        print("  ", p)

## 10. Flood impact demonstration (plan 11)XLRS-Bench is **not** a flood dataset, so this runs on imagery you supply via a smallmanifest. There is no ground truth unless you provide it, so no accuracy is computedand none is reported - the demonstration is that the answer is traceable to specificdetected objects, and that the relation block supplies the object-to-hazard links thatspan different parts of the scene.```json[  {    "id": "scene_a",    "image": "data/flood/scene_a.tif",    "questions": ["Which schools are located inside the flooded area?"]  }]``````bashpython scripts/05_flood_case_study.py --config configs/main.yaml \    --manifest data/flood/manifest.json --out results/flood --overlay```

In [ ]:
FLOOD_MANIFEST = "data/flood/manifest.json"if Path(FLOOD_MANIFEST).exists():    res = subprocess.run(        [sys.executable, "scripts/05_flood_case_study.py", "--config", CONFIG,         "--manifest", FLOOD_MANIFEST, "--out", "results/flood", "--overlay"],        capture_output=True, text=True)    print(res.stdout[-4000:])    report = Path("results/flood/flood_report.md")    if report.exists():        from IPython.display import Markdown, display        display(Markdown(report.read_text(encoding="utf-8")[:6000]))else:    print("No flood manifest at {}.".format(FLOOD_MANIFEST))    print("This section is optional - the main experiment does not depend on it.")

## 11. Wrap upEverything needed for the manuscript is under the report directory:- `table1_main` ... `table7_error_taxonomy` - CSV for analysis, LaTeX for the paper  (LaTeX tables need `\usepackage{booktabs}`)- `fig1` ... `fig7` - PNG for drafts, PDF (vector) for the camera-ready- `aggregate.json` - every number in the tables, for re-analysis- `qualitative/` - scene-graph overlays and per-case markdown traces**Before writing any number into the paper, confirm:**1. `backend_is_stub` is `False` in every run manifest.2. `n_common` equals the number of samples you intended to evaluate.3. Baseline 3 and Proposed have equal mean VLM calls and megapixels (section 5.1).4. The relational partition you report is the one you fixed *before* seeing results.5. Parse-failure rates are similar across arms - a large difference means you are   partly measuring format compliance rather than reasoning.

In [ ]:
from ctsr.utils.io_utils import atomic_write_json, read_jsonprint("Run manifests:")for arm_dir in sorted(p for p in RESULTS_DIR.iterdir() if p.is_dir()):    m = read_json(arm_dir / "manifest.json", default=None)    if not m:        continue    print("  {:<16} stub={:<6} n={:<5} status={} hash={}".format(        m.get("run_name"), str(m.get("backend_is_stub")),        m.get("n_samples_total"), m.get("status"), m.get("config_hash")))atomic_write_json(REPORT_DIR / "notebook_aggregate.json", agg)print("\nSaved:", REPORT_DIR / "notebook_aggregate.json")try:    backend.unload()    print("Model unloaded; GPU memory released.")except Exception:    pass